In [14]:
import os
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from pprint import pprint

from utils.analyses.output_data_preprocess import *
from utils.analyses.descriptives import *
from utils.analyses.ref_letters_analysis import *
from utils.analyses.psychometrics import *

In [15]:
# set all needed directories
current_dir = os.getcwd()
output_data_dir = os.path.join(current_dir, "output_data")
results_dir = os.path.join("C:/Users/jana/Documents/Repos/measuring-sexism-in-LLMs", "results")

In [16]:
def preprocess_test_data(task):
    df = load_and_concat_jsons(base_dir=output_data_dir, subfolder=task, file_suffix=task)

    # count NaN answers for every model
    nan_counts = df.groupby('model_name')['answer_reversed'].apply(lambda x: x.isna().sum()).reset_index()
    nan_counts.columns = ['model_name', 'nan_count']
    # save in json
    nan_counts.to_json(os.path.join(results_dir,task,"nan_count_per_model.json"), orient="columns", indent=2)

    # for SR2K: transform answers to item 3 from scale 1-3 to scale 1-4
    if task == "SR2K":
        item_3 = df["item_id"] == 3
        df.loc[item_3, "answer_reversed"] = 1.5 * df.loc[item_3, "answer_reversed"] - 0.5
        # reverse all scores such that hich value = high racism
        df["answer_reversed"] = 4 - df["answer_reversed"] + 1
    # calculate mean and sd over the different seeds for each item
    #df_agg = df.groupby(["model_name", "seeds", "item_id", "subscale", "reversed"], as_index=False).agg(avg_answer_reversed=('answer_reversed', 'mean'))

    avg_scores = (
        df.groupby(["model_name", "seed"])["answer_reversed"]
        .mean()
        .reset_index(name="total")
    )

    return avg_scores

In [17]:
def r_by_seed(merged:pd.DataFrame, construct:str):
    results = []

    for seed, group in merged.groupby("seed"):
        # compute the correlation for this seed only
        r, lower, upper = spearman_rank_corr(
            group[f"{construct}_score"], 
            group["total"], 
            alternative="greater"
        )
        
        results.append({
            "seed": seed,
            "spearman_r": r,
            "lower_CI": lower,
            "upper_CI": upper
        })

    pprint(results)

    return results

In [18]:
def r_avg_std(results: dict):
    # Extract Spearman correlations
    r_values = [item["spearman_r"].statistic for item in results]

    # Apply Fisher z-transformation
    z_values = np.arctanh(r_values)  # arctanh is Fisher z

    # Compute mean and std in z-space
    mean_z = np.mean(z_values)
    std_z = np.std(z_values, ddof=1)  # sample std

    # Transform mean back to correlation
    avg_r = np.tanh(mean_z)

    # Optional: approximate std in correlation space (not exact, but indicative)
    std_r = np.tanh(mean_z + std_z) - avg_r  # upper deviation

    print(f"Average Spearman correlation across seeds (Fisher z): {avg_r:.4f} (approx. std: {std_r:.4f})")

# Sexism

In [19]:
df_ASI = preprocess_test_data("ASI")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\ASI


In [20]:
df_ref = load_and_concat_jsons(base_dir=output_data_dir, subfolder="ref_letter_generation", file_suffix="")

df_ref_wide = df_ref.groupby(["model_name", "seed"]).apply(
    analyze_ref_letters,
    include_groups = False
).reset_index()

# get all columns containing OR values
OR_columns = [col for col in df_ref_wide.columns if "OR" in col]

# calculate overall sexism score for each context by averaging over OR values
df_ref_wide["sexism_score"] = df_ref_wide[OR_columns].mean(axis=1)

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\ref_letter_generation


In [21]:
summary = df_ref_wide.groupby('model_name')['sexism_score'].agg(['mean', 'std']).reset_index()
print(summary)

                     model_name      mean       std
0        Llama-3.1-70B-Instruct  1.699298  0.062322
1         Llama-3.1-8B-Instruct  1.651996  0.084170
2         Llama-3.1-Centaur-70B  1.182177  0.223536
3        Llama-3.3-70B-Instruct  1.658384  0.081476
4      Mistral-7B-Instruct-v0.3  1.756308  0.136340
5   Mistral-Large-Instruct-2411  1.581470  0.059480
6          Qwen2.5-14B-Instruct  1.680467  0.117502
7          Qwen2.5-32B-Instruct  1.683739  0.095235
8          Qwen2.5-72B-Instruct  1.620577  0.056617
9           Qwen2.5-7B-Instruct  1.656764  0.071401
10       Qwen3-4B-Instruct-2507  1.650762  0.066041
11             gemini-2.5-flash  1.753637  0.055796
12               gemini-2.5-pro  1.513240  0.054705
13               gemma-3-12b-it  1.560304  0.044569
14                gemma-3-1b-it  1.426233  0.065779
15               gemma-3-27b-it  1.600519  0.109807
16                gemma-3-4b-it  1.507865  0.085493


In [22]:
merged_sexism = pd.merge(left=df_ASI, right=df_ref_wide[["model_name","seed", "sexism_score"]], how="left", on=["model_name", "seed"])
merged_sexism = merged_sexism[merged_sexism.model_name != "Llama-3.1-Centaur-70B"]

In [23]:
results_ASI = r_by_seed(merged_sexism, construct="sexism")

[{'lower_CI': -0.6685419628831057,
  'seed': 1,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.25849589102417725), pvalue=np.float64(0.8331467616397665)),
  'upper_CI': 0.2720814144476101},
 {'lower_CI': -0.5514437787686759,
  'seed': 2,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.07669649888473705), pvalue=np.float64(0.611149276897146)),
  'upper_CI': 0.4355767498863793},
 {'lower_CI': -0.784732733129049,
  'seed': 3,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.4730292428880248), pvalue=np.float64(0.9678803111417037)),
  'upper_CI': 0.02963190935203246},
 {'lower_CI': -0.5845566812017746,
  'seed': 4,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.1250920132623678), pvalue=np.float64(0.6778158176764153)),
  'upper_CI': 0.39512227245880466},
 {'lower_CI': -0.5482273676108528,
  'seed': 5,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.07211186646889438), pvalue=np.float64(0.6046470816043014)),
  'upper_CI': 0.43930472707

In [24]:
r_avg_std(results_ASI)

Average Spearman correlation across seeds (Fisher z): -0.2076 (approx. std: 0.1835)


# Racism

In [25]:
df_SR2K = preprocess_test_data("SR2K")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\SR2K


In [34]:
df_hr = pd.read_csv(os.path.join(results_dir, "ecological", "housing_per_model_seed.csv"))
df_hr = df_hr.rename(columns={"mean_difference":"racism_score"})

In [35]:
summary = df_hr.groupby('model_name')['racism_score'].agg(['mean', 'std']).reset_index()
print(summary)

                     model_name      mean       std
0        Llama-3.1-70B-Instruct  2.408112  0.291298
1         Llama-3.1-8B-Instruct  2.343116  0.171365
2        Llama-3.3-70B-Instruct  2.887573  0.125722
3      Mistral-7B-Instruct-v0.3  1.196099  0.098689
4   Mistral-Large-Instruct-2411  2.134824  0.204055
5          Qwen2.5-14B-Instruct  1.825804  0.078536
6          Qwen2.5-32B-Instruct  2.655580  0.167873
7          Qwen2.5-72B-Instruct  3.208634  0.074096
8           Qwen2.5-7B-Instruct  0.859179  0.110658
9        Qwen3-4B-Instruct-2507  1.062107  0.091814
10             gemini-2.5-flash  2.468276  0.132353
11               gemini-2.5-pro  2.500438  0.163988
12               gemma-3-12b-it  2.564266  0.137784
13                gemma-3-1b-it  0.005326  0.308398
14               gemma-3-27b-it  2.374130  0.104078
15                gemma-3-4b-it  1.436958  0.172941


In [28]:
merged_racism = pd.merge(left=df_SR2K, right=df_hr[["model_name","seed", "racism_score"]], how="left", on=["model_name", "seed"])
merged_racism = merged_racism[merged_racism.model_name != "Llama-3.1-Centaur-70B"]

In [29]:
results_SR2K = r_by_seed(merged_racism, construct="racism")

[{'lower_CI': -0.8233961455887346,
  'seed': 1,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.5536762384726116), pvalue=np.float64(0.9869630156892998)),
  'upper_CI': -0.07989046577321922},
 {'lower_CI': -0.8258157361824172,
  'seed': 2,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.5588972211855656), pvalue=np.float64(0.9877931716006573)),
  'upper_CI': -0.08739826987735363},
 {'lower_CI': -0.843217872954053,
  'seed': 3,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.5970814340265321), pvalue=np.float64(0.9926974882921524)),
  'upper_CI': -0.14398554917057474},
 {'lower_CI': -0.8972761589286622,
  'seed': 4,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.7232655393400694), pvalue=np.float64(0.9992282192649796)),
  'upper_CI': -0.35473739382779956},
 {'lower_CI': -0.8288691766517837,
  'seed': 5,
  'spearman_r': SignificanceResult(statistic=np.float64(-0.5655162653276263), pvalue=np.float64(0.9887865540572516)),
  'upper_CI': -0.096994

In [30]:
r_avg_std(results_SR2K)

Average Spearman correlation across seeds (Fisher z): -0.6042 (approx. std: 0.0837)


# Morality

In [31]:
df_MFQ = preprocess_test_data("MFQ")

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\MFQ


In [32]:
df_advice = load_and_concat_jsons(base_dir=output_data_dir, subfolder="advice", file_suffix="")
df_advice = df_advice[df_advice["model_name"] != "Llama-3.1-Centaur-70B"]

#print(df_advice['pro_value'].value_counts())

condition_match = (
    (df_advice['pro_value'] == True) & (df_advice['judge_action_taken'] == 'yes')
) | (
    (df_advice['pro_value'] == False) & (df_advice['judge_action_taken'] == 'no')
)
df_advice['match'] = np.where(condition_match, 1,0)

df_advice_agg = (
        df_advice.groupby(["model_name", "seed", "subscale"])["match"]
        .mean()
        .reset_index(name="subscale_score")
    )

c:\Users\jana\Documents\Repos\measuring-sexism-in-LLMs\src\output_data\advice


In [33]:
subscales = ["authority", "care", "fairness", "ingroup", "purity"]

for sub in subscales:
    print("-----------------------------------------")
    print(sub)
    df_advice_agg_sub = df_advice_agg.loc[df_advice_agg["subscale"] == sub]
    summary = df_advice_agg_sub.groupby('model_name')['subscale_score'].agg(['mean', 'std']).reset_index()
    print(summary)

    merged_sub = pd.merge(left=df_SR2K, right=df_advice_agg_sub[["model_name","seed", "subscale_score"]], how="left", on=["model_name", "seed"])
    merged_sub = merged_sub[merged_sub.model_name != "Llama-3.1-Centaur-70B"]

    results_sub = r_by_seed(merged_sub, construct="subscale")

    r_avg_std(results_sub)
    

-----------------------------------------
authority
                     model_name      mean       std
0        Llama-3.1-70B-Instruct  0.280000  0.007454
1         Llama-3.1-8B-Instruct  0.316667  0.016667
2        Llama-3.3-70B-Instruct  0.340000  0.030277
3      Mistral-7B-Instruct-v0.3  0.276667  0.034561
4   Mistral-Large-Instruct-2411  0.266667  0.042492
5          Qwen2.5-14B-Instruct  0.260000  0.053489
6          Qwen2.5-32B-Instruct  0.250000  0.031180
7          Qwen2.5-72B-Instruct  0.266667  0.026352
8           Qwen2.5-7B-Instruct  0.283333  0.011785
9        Qwen3-4B-Instruct-2507  0.246667  0.021731
10             gemini-2.5-flash  0.320000  0.013944
11               gemini-2.5-pro  0.340000  0.043461
12               gemma-3-12b-it  0.246667  0.044721
13                gemma-3-1b-it  0.226667  0.030277
14               gemma-3-27b-it  0.250000  0.011785
15                gemma-3-4b-it  0.260000  0.025276
[{'lower_CI': -0.7506972186620896,
  'seed': 1,
  'spearman_r': 

# Convergent